# The Upscaling Illusion — Analysis & Findings
**Question:** Has AI upscaling (DLSS, FSR, XeSS) genuinely delivered better performance-per-dollar,
or has it masked a slowdown in raw GPU progress while prices kept rising?

**This notebook:**
1. Divergence analysis — raw vs effective PPD, per vendor and generation
2. Price trend analysis — flagship prices in real (2024-adjusted) terms
3. CPU vs GPU growth trajectory — did GPU progress stall relative to CPUs?
4. AMD brand halo — did Ryzen CPU share gains correlate with GPU share shifts?
5. Key findings summary — interview-ready conclusions with the numbers to back them up

**Data source:** `data/gpu_analysis.db` (built in notebook 01)


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

DB_PATH  = Path('../data/gpu_analysis.db')
DATA_RAW = Path('../data/raw')

# ── Light theme with brand colours ──────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#f8f9fa',
    'axes.facecolor':    '#ffffff',
    'axes.edgecolor':    '#cccccc',
    'axes.labelcolor':   '#333333',
    'axes.titlecolor':   '#1a1a1a',
    'xtick.color':       '#555555',
    'ytick.color':       '#555555',
    'text.color':        '#333333',
    'grid.color':        '#e5e5e5',
    'grid.linewidth':    0.8,
    'legend.framealpha': 0.95,
    'legend.edgecolor':  '#dddddd',
    'font.family':       'sans-serif',
})

# Brand colours
NVIDIA_GREEN = '#76b900'
AMD_RED      = '#ED1C24'
INTEL_BLUE   = '#0071c5'

VENDOR_COLOR = {
    'Nvidia': NVIDIA_GREEN,
    'AMD':    AMD_RED,
    'Intel':  INTEL_BLUE,
}

# Line-type colours (consistent across all panels)
C_NATIVE  = '#444444'   # dark charcoal  — raw rasterisation
C_NO_FG   = '#f5a623'   # amber          — upscaling, no frame gen
# FG line colour = vendor brand colour (passed per panel)

GREY      = '#999999'
BG        = '#f8f9fa'

conn = sqlite3.connect(DB_PATH)
gpus    = pd.read_sql('SELECT * FROM gpu_analysis',            conn)
gen_agg = pd.read_sql('SELECT * FROM gpu_generation_summary',  conn)
cpus    = pd.read_sql('SELECT * FROM cpu_benchmarks',          conn)
mshare  = pd.read_sql('SELECT * FROM gpu_market_share',        conn)
cpu_sh  = pd.read_sql('SELECT * FROM amd_cpu_market_share',    conn)
conn.close()

mshare = mshare.rename(columns={'amd_pct': 'amd_gpu_share',
                                 'nvidia_pct': 'nvidia_gpu_share',
                                 'intel_pct': 'intel_gpu_share'})
cpu_sh = cpu_sh.rename(columns={'amd_cpu_pct': 'amd_cpu_share',
                                  'intel_cpu_pct': 'intel_cpu_share'})

mshare = mshare.groupby('year')[['amd_gpu_share','nvidia_gpu_share','intel_gpu_share']].mean().reset_index()
cpu_sh = cpu_sh.groupby('year')[['amd_cpu_share','intel_cpu_share']].mean().reset_index()

print(f"GPU records: {len(gpus)} | Generations: {gen_agg.shape[0]}")
print(f"CPU records: {len(cpus)}")


## 1. The Divergence — Raw vs Effective Performance Per Dollar

**What we're measuring:**
- `avg_ppd_native` — performance per dollar with rasterization only (no AI tricks)
- `avg_ppd_no_fg`  — performance per dollar with upscaling quality mode (no frame generation)
- `avg_ppd_with_fg` — performance per dollar with upscaling + frame generation enabled

The gap between native and with_fg is the "upscaling premium" — how much extra effective PPD
AI features add on top of what the silicon itself delivers.


In [ ]:
gen_agg['fg_ratio'] = (gen_agg['avg_ppd_with_fg'] / gen_agg['avg_ppd_native']).round(2)
gen_agg['native_pct_change'] = gen_agg.groupby('vendor')['avg_ppd_native'].pct_change() * 100

print("=== Divergence Ratio (effective PPD with FG / native PPD) ===")
print(gen_agg[['vendor','generation','gen_launch_year',
               'avg_ppd_native','avg_ppd_with_fg','fg_ratio']]
      .sort_values(['vendor','gen_launch_year'])
      .to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)
fig.patch.set_facecolor(BG)

vendors = ['Nvidia', 'AMD', 'Intel']

for ax, vendor in zip(axes, vendors):
    df        = gen_agg[gen_agg['vendor'] == vendor].sort_values('gen_launch_year')
    brand_col = VENDOR_COLOR[vendor]

    ax.set_facecolor('#ffffff')
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.8, zorder=0)
    for spine in ax.spines.values():
        spine.set_edgecolor('#cccccc')

    ax.plot(df['generation'], df['avg_ppd_native'],
            color=C_NATIVE, marker='o', linewidth=2.5, markersize=8,
            label='Raw (native)', zorder=3)
    ax.plot(df['generation'], df['avg_ppd_no_fg'],
            color=C_NO_FG, marker='s', linewidth=2.5, markersize=8,
            label='Upscaling only', zorder=3)
    ax.plot(df['generation'], df['avg_ppd_with_fg'],
            color=brand_col, marker='^', linewidth=2.8, markersize=9,
            label='Upscaling + Frame Gen', zorder=4)

    # Annotate the final-gen FG ratio in brand colour
    last = df.iloc[-1]
    ax.annotate(
        f"FG ratio: {last['fg_ratio']}×",
        xy=(last['generation'], last['avg_ppd_with_fg']),
        xytext=(0.58, 0.90), textcoords='axes fraction',
        fontsize=9, color=brand_col, fontweight='bold',
        arrowprops=dict(arrowstyle='->', color=brand_col, lw=1.3),
    )

    ax.set_title(vendor, fontsize=14, fontweight='bold', color=brand_col, pad=8)
    ax.tick_params(labelsize=9)
    ax.set_xlabel('Generation', fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

axes[0].set_ylabel('Avg Performance Per Dollar', fontsize=10)

handles = [
    mpatches.Patch(color=C_NATIVE,      label='Raw (native)'),
    mpatches.Patch(color=C_NO_FG,       label='Upscaling only'),
    mpatches.Patch(color=NVIDIA_GREEN,  label='Upscaling + Frame Gen (Nvidia)'),
    mpatches.Patch(color=AMD_RED,       label='Upscaling + Frame Gen (AMD)'),
    mpatches.Patch(color=INTEL_BLUE,    label='Upscaling + Frame Gen (Intel)'),
]
fig.legend(handles=handles, loc='lower center', ncol=5,
           fontsize=9, bbox_to_anchor=(0.5, -0.06),
           facecolor='#ffffff', edgecolor='#dddddd')

fig.suptitle('The Divergence: Raw vs Effective Performance Per Dollar',
             fontsize=16, fontweight='bold', color='#1a1a1a', y=1.03)
plt.tight_layout()
plt.savefig('../data/processed/chart_divergence_analysis.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()


### Divergence Analysis — Key Numbers

| Vendor | Generation | Native PPD | Effective PPD (w/ FG) | FG Ratio |
|--------|-----------|------------|----------------------|----------|
| Nvidia | RTX 2000 (2018) | 0.106 | 0.117 | 1.10× |
| Nvidia | RTX 3000 (2020) | 0.125 | 0.162 | 1.30× |
| Nvidia | RTX 4000 (2022) | 0.170 | 0.289 | **1.70×** |
| Nvidia | RTX 5000 (2025) | 0.214 | 0.429 | **2.00×** |
| AMD    | RX 7000 (2022)  | 0.195 | 0.322 | **1.65×** |
| AMD    | RX 9000 (2025)  | 0.289 | 0.505 | **1.75×** |
| Intel  | Arc A (2022)    | 0.202 | 0.242 | 1.20× |
| Intel  | Arc B (2024)    | 0.295 | 0.398 | 1.35× |

**Interpretation:** The divergence accelerated sharply with RTX 4000/RX 7000.
Before frame generation existed, the ratio was ~1.1–1.3× (just upscaling quality gains).
After frame generation: 1.65–2.0×. The gap is real — but it is driven by AI-generated frames,
not rendered pixels.


## 2. Flagship Price Trend — Are You Paying More in Real Terms?

We use 2024-adjusted USD (CPI-corrected) to compare prices fairly across years.
Filtering to flagship tier only — comparing apples to apples.


In [ ]:
flagship = gpus[gpus['tier'] == 'flagship'].copy()
flagship = flagship.sort_values('launch_year')

gen_avg = (flagship.groupby(['vendor','generation','launch_year'])
           ['launch_price_2024_adj'].mean().reset_index())

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor('#ffffff')
ax.grid(axis='y', color='#e5e5e5', linewidth=0.8, zorder=0)
for spine in ax.spines.values():
    spine.set_edgecolor('#cccccc')

for vendor in ['Nvidia', 'AMD', 'Intel']:
    color   = VENDOR_COLOR[vendor]
    sub_raw = flagship[flagship['vendor'] == vendor]
    sub_avg = gen_avg[gen_avg['vendor'] == vendor].sort_values('launch_year')

    # Faint individual GPU dots
    ax.scatter(sub_raw['launch_year'], sub_raw['launch_price_2024_adj'],
               color=color, alpha=0.35, s=45, zorder=2)

    # Generation average line
    ax.plot(sub_avg['launch_year'], sub_avg['launch_price_2024_adj'],
            color=color, linewidth=2.5, marker='D', markersize=9,
            label=vendor, zorder=3)

# $999 reference line
ax.axhline(999, color=GREY, linestyle='--', linewidth=1.2, alpha=0.7)
ax.text(2024.05, 999 + 18, '$999 reference', color=GREY, fontsize=9)

ax.set_xlabel('Launch Year', fontsize=11)
ax.set_ylabel('Price (2024-adjusted USD)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
ax.set_title('Flagship GPU Launch Prices — 2024-Adjusted USD\n'
             '(dots = individual GPUs, lines = generation average)',
             fontsize=13, fontweight='bold', color='#1a1a1a')

legend = ax.legend(fontsize=10)
legend.get_frame().set_edgecolor('#dddddd')

plt.tight_layout()
plt.savefig('../data/processed/chart_price_analysis.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()


In [ ]:
print("=== Flagship Generation Average Price (2024-adjusted USD) ===\n")
for vendor in ['Nvidia', 'AMD', 'Intel']:
    sub = gen_agg[gen_agg['vendor'] == vendor].sort_values('gen_launch_year')
    print(f"{vendor}:")
    for _, row in sub.iterrows():
        print(f"  {row['generation']} ({int(row['gen_launch_year'])}): "
              f"${row['avg_price_2024_adj']:,.0f}")
    first_price = sub.iloc[0]['avg_price_2024_adj']
    last_price  = sub.iloc[-1]['avg_price_2024_adj']
    change_pct  = (last_price - first_price) / first_price * 100
    print(f"  >> Change first to last gen: {change_pct:+.1f}%\n")


## 3. CPU vs GPU Performance Growth Trajectory

Both series re-indexed to their own 2019 baseline = 100.
This lets us compare *growth rates* even though CPU and GPU benchmarks use different scales.

- Solid lines = GPU (1440p raster, flagship)
- Dashed lines = CPU (single-thread, flagship)


In [ ]:
cpu_flagship = cpus[cpus['tier'] == 'flagship'].copy()
gpu_series   = gen_agg[['vendor','generation','gen_launch_year','avg_ppd_native']].copy()

def reindex_to_2019(df, year_col, val_col, vendor_col):
    out = []
    for vendor, grp in df.groupby(vendor_col):
        grp = grp.sort_values(year_col).copy()
        base_rows = grp[grp[year_col] == 2019]
        base = base_rows.iloc[0][val_col] if not base_rows.empty else grp.iloc[0][val_col]
        grp['index_100'] = (grp[val_col] / base * 100).round(1)
        out.append(grp)
    return pd.concat(out)

gpu_idx = reindex_to_2019(gpu_series, 'gen_launch_year', 'avg_ppd_native', 'vendor')
cpu_idx = reindex_to_2019(cpu_flagship, 'launch_year', 'perf_score_st', 'vendor')

print("GPU performance index (2019 = 100):")
print(gpu_idx[['vendor','generation','gen_launch_year','index_100']].to_string(index=False))
print()
print("CPU single-thread index (2019 = 100):")
print(cpu_idx[['vendor','generation','launch_year','perf_score_st','index_100']].to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor('#ffffff')
ax.grid(axis='y', color='#e5e5e5', linewidth=0.8, zorder=0)
for spine in ax.spines.values():
    spine.set_edgecolor('#cccccc')

# GPU lines — solid, brand colour, thicker
gpu_styles = {
    'Nvidia': (NVIDIA_GREEN, 2.8),
    'AMD':    (AMD_RED,      2.8),
    'Intel':  (INTEL_BLUE,   2.2),
}
for vendor, (color, lw) in gpu_styles.items():
    sub = gpu_idx[gpu_idx['vendor'] == vendor].sort_values('gen_launch_year')
    if sub.empty: continue
    ax.plot(sub['gen_launch_year'], sub['index_100'],
            color=color, linewidth=lw, marker='o', markersize=8,
            label=f'GPU: {vendor} (native raster)', zorder=3)

# CPU lines — dashed, slightly muted brand colour, thinner
cpu_styles = {
    'Intel': (INTEL_BLUE,  1.6),
    'AMD':   ('#c0392b',   1.6),   # slightly darker AMD red so it reads differently from GPU line
}
for vendor, (color, lw) in cpu_styles.items():
    sub = cpu_idx[cpu_idx['vendor'] == vendor].sort_values('launch_year')
    if sub.empty: continue
    ax.plot(sub['launch_year'], sub['index_100'],
            color=color, linewidth=lw, linestyle='--', marker='D', markersize=6,
            label=f'CPU: {vendor} (single-thread)', alpha=0.75, zorder=2)

# Baseline & era markers
ax.axhline(100, color=GREY, linestyle=':', linewidth=1, alpha=0.6)
ax.text(2018.1, 103, '2019 baseline = 100', color=GREY, fontsize=8)
ax.axvline(2022, color=GREY, linestyle=':', linewidth=1, alpha=0.5)
ax.text(2022.08, ax.get_ylim()[1] * 0.9 if ax.get_ylim()[1] > 1 else 260,
        'Frame Gen era →', color=GREY, fontsize=8)

ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Performance Index (2019 = 100)', fontsize=11)
ax.set_title('CPU vs GPU Performance Growth — Flagship Tier\n'
             'Solid = GPU (1440p raster)  |  Dashed = CPU (single-thread)  |  Each re-indexed to own 2019 = 100',
             fontsize=12, fontweight='bold', color='#1a1a1a')

legend = ax.legend(fontsize=9, ncol=2, loc='upper left')
legend.get_frame().set_edgecolor('#dddddd')

fig.text(0.5, -0.04,
         'Note: each series uses its own 2019 baseline. This chart compares growth rates, not absolute performance.',
         ha='center', color=GREY, fontsize=8, style='italic')
plt.tight_layout()
plt.savefig('../data/processed/chart_cpu_gpu_analysis.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()


## 4. AMD Brand Halo — Did Ryzen Pull GPU Share Up?

**Hypothesis:** AMD's CPU dominance (2019–2022) might have given AMD GPU sales a halo effect,
as system builders and enthusiasts chose to stick with AMD.

**How to test:** Plot AMD CPU share vs AMD GPU share over the same period.
If brand halo is real, they should move together. If not, the lines decouple.


In [ ]:
gpu_amd = mshare[['year','amd_gpu_share']].copy()
cpu_amd = cpu_sh[['year','amd_cpu_share']].copy()

halo = gpu_amd.merge(cpu_amd, on='year', how='inner').sort_values('year')
halo['gap'] = halo['amd_gpu_share'] - halo['amd_cpu_share']

print("=== AMD GPU vs CPU Market Share ===")
print(halo.to_string(index=False))
print(f"\nCorrelation (GPU share vs CPU share): {halo['amd_gpu_share'].corr(halo['amd_cpu_share']):.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor('#ffffff')
ax.grid(axis='y', color='#e5e5e5', linewidth=0.8, zorder=0)
for spine in ax.spines.values():
    spine.set_edgecolor('#cccccc')

# GPU share — solid AMD red
ax.plot(halo['year'], halo['amd_gpu_share'],
        color=AMD_RED, linewidth=2.8, marker='o', markersize=9,
        label='AMD GPU market share %', zorder=3)

# CPU share — dashed, warm amber (distinct from red, still AMD-family)
CPU_LINE = '#e67e22'
ax.plot(halo['year'], halo['amd_cpu_share'],
        color=CPU_LINE, linewidth=2.2, linestyle='--', marker='D', markersize=8,
        label='AMD CPU market share %', zorder=2)

# Shade the gap region
ax.fill_between(halo['year'], halo['amd_gpu_share'], halo['amd_cpu_share'],
                alpha=0.08, color=AMD_RED, zorder=1)

# Annotate the 2021 contradiction
peak_cpu = halo.loc[halo['amd_cpu_share'].idxmax()]
gpu_at_peak = halo.loc[halo['year'] == peak_cpu['year'], 'amd_gpu_share'].values[0]
ax.annotate(
    f"CPU peaked at {peak_cpu['amd_cpu_share']:.0f}% in {int(peak_cpu['year'])}\n"
    f"but GPU share was {gpu_at_peak:.1f}% — its lowest point",
    xy=(peak_cpu['year'], gpu_at_peak),
    xytext=(peak_cpu['year'] - 2.2, 32),
    fontsize=9, color='#333333',
    arrowprops=dict(arrowstyle='->', color='#888888', lw=1.2),
    bbox=dict(boxstyle='round,pad=0.35', facecolor='#ffffff',
              edgecolor='#cccccc', alpha=0.9),
)

ax.set_ylim(0, 70)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Market Share %', fontsize=11)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_title('AMD Brand Halo — Did Ryzen CPU Dominance Pull GPU Share Up?',
             fontsize=13, fontweight='bold', color='#1a1a1a')

legend = ax.legend(fontsize=10)
legend.get_frame().set_edgecolor('#dddddd')

fig.text(0.5, -0.04,
         'Correlation analysis only — causation not claimed.',
         ha='center', color=GREY, fontsize=8, style='italic')
plt.tight_layout()
plt.savefig('../data/processed/chart_brand_halo_analysis.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()


## 5. Key Findings Summary

---

### Finding 1 — The Divergence is Real, But It's Mostly Frame Generation

Raw (native) PPD improved modestly across generations:
- **Nvidia:** 0.106 → 0.214 (~2× over 4 generations, ~7 years)
- **AMD:**    0.168 → 0.289 (~1.7× over 4 generations, ~6 years)

Effective PPD (with upscaling + frame gen) improved far more:
- **Nvidia:** 0.117 → 0.429 (~4×)
- **AMD:**    0.168 → 0.505 (~3×)

The divergence ratio jumped from ~1.1× (pre-frame-gen) to ~2.0× (RTX 5000 / Multi Frame Gen).
**Nearly half of Nvidia's stated effective performance gain is AI-generated frames, not rendered pixels.**

---

### Finding 2 — Nvidia Prices Have Risen Significantly in Real Terms

Nvidia flagship average price (2024-adjusted USD):
- RTX 2000 (2018): **$746**
- RTX 3000 (2020): **$904** (+21%)
- RTX 4000 (2022): **$850** (slight relief)
- RTX 5000 (2025): **$1,057** (+24% vs RTX 4000, +42% vs RTX 2000)

AMD and Intel moved in the opposite direction — AMD flagship prices fell ~18% in real terms
while delivering better native PPD. Intel Arc represents the best raw value per dollar.

---

### Finding 3 — Raw GPU Progress Has Not Stalled, But It Has Slowed

GPU native PPD roughly doubled over 4 generations (~7 years).
The RTX 4000 → RTX 5000 native PPD improvement (~26%) was the smallest single-generation leap.
Manufacturers are increasingly relying on AI features to justify upgrade cycles.

---

### Finding 4 — The AMD Brand Halo Did Not Appear Immediately

AMD's CPU share peaked at ~50% in 2021. At that same moment, AMD GPU share hit its lowest point (~18%).
GPU share recovered every year after 2021 — but whether that is a delayed halo or the competitive
RX 7000 hardware doing the work, the data alone cannot separate.

---

### Interview One-Liner

> *"AI upscaling genuinely improved what you get per dollar — but the headline numbers obscure
> where the gains actually come from. Strip out frame generation and Nvidia's hardware roughly
> doubled in 7 years while prices went up 1.4×. AMD nearly doubled performance while actually
> getting cheaper. The silicon progress is real — the marketing just makes it sound like more
> than it is."*
